In [ ]:
#pip install langchain

In [ ]:
#!pip install -U langchain-community docx2txt

In [33]:
from langchain_community.document_loaders import Docx2txtLoader

files = {
    "2023": "data/2023_Annual_Report.docx",
    "2024": "data/2024_Annual_Report.docx",
    "2025": "data/2025_AnnualReport.docx"
}

all_documents = []

for year, file_path in files.items():

    loader = Docx2txtLoader(file_path)
    documents = loader.load()

    # Add year and source as metadata
    for doc in documents:
        doc.metadata["year"] = year
        doc.metadata["source"] = file_path

    all_documents.extend(documents)

print("Total documents loaded:", len(all_documents))

Total documents loaded: 3


In [35]:
print(all_documents[0].page_content[:2000])

Dear shareholders, colleagues, customers, and partners, 

We are living through a time of historic challenge and opportunity. As I write this, the world faces ongoing economic, social, and geopolitical volatility. At the same time, we have entered a new age of AI that will fundamentally transform productivity for every individual, organization, and industry on earth, and help us address some of our most pressing challenges. 

This next generation of AI will reshape every software category and every business, including our own. Forty-eight years after its founding, Microsoft remains a consequential company because time and time again—from PC/Server, to Web/Internet, to Cloud/Mobile—we have adapted to technological paradigm shifts. Today, we are doing so once again, as we lead this new era. 

Amid this transformation, our mission to empower every person and every organization on the planet to achieve more remains constant. As a company, we believe we can be the democratizing force for th

In [36]:
print(all_documents[0].metadata)

{'source': 'data/2023_Annual_Report.docx', 'year': '2023'}


In [37]:
for doc in all_documents:
    print("Year:", doc.metadata["year"])
    print("Characters:", len(doc.page_content))
    print("-" * 50)

Year: 2023
Characters: 291725
--------------------------------------------------
Year: 2024
Characters: 290613
--------------------------------------------------
Year: 2025
Characters: 260226
--------------------------------------------------


In [38]:
print(all_documents[1].page_content[:2000])
print(all_documents[1].metadata)

Dear shareholders, colleagues, customers, and partners:

Fiscal year 2024 was a pivotal year for Microsoft. We entered our 50th year as a company and the second year of the AI platform shift. With these milestones, I’ve found myself reflecting on how Microsoft has remained a consequential company decade after decade in an industry with no franchise value. And I realize that it’s because—time and time again, when tech paradigms have shifted—we have seized the opportunity to reinvent ourselves to stay relevant to our customers, our partners, and our employees. And that’s what we are doing again today. 

Microsoft has been a platform and tools company from the start. We were founded in 1975 with a belief in creating technology that would enable others to create their own. And, nearly 50 years later, this belief remains at the heart of our mission to empower every person and every organization on the planet to achieve more. 

This year, we moved from talking about AI to helping our custome

# step 2 chunking


In [39]:
#We use RecursiveCharacterTextSplitter because financial reports contain different types of text, and we want to split them without breaking the meaning too aggressively.

In [40]:
#!pip install -U langchain-text-splitters

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [42]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [43]:
chunks = text_splitter.split_documents(all_documents)

print("Original documents:", len(all_documents))
print("Total chunks:", len(chunks))

Original documents: 3
Total chunks: 1143


In [44]:
print(chunks[1].page_content)

Amid this transformation, our mission to empower every person and every organization on the planet to achieve more remains constant. As a company, we believe we can be the democratizing force for this new generation of technology and the opportunity it will help unlock for every country, community, and individual, while mitigating its risks. 

Here are just a few examples of how we are already doing this: 

•	Leading electronic health records vendor Epic is addressing some of the biggest challenges facing the healthcare industry today—including physician burnout—by deploying a wide range of copilot solutions built on Azure OpenAI Service and Dragon Ambient eXperience Copilot. 

•	Mercado Libre is reducing the time its developers spend writing code by more than 50 percent with GitHub Copilot, as the company works to democratize e-commerce across Latin America.


In [45]:

print(chunks[1000].metadata)

{'source': 'data/2025_AnnualReport.docx', 'year': '2025'}


In [46]:
print(chunks[0].metadata)

{'source': 'data/2023_Annual_Report.docx', 'year': '2023'}


In [47]:
for i in range(3):
    print(f"\n--- Chunk {i+1} ---")
    print(chunks[i].page_content[:500])
    print("Year:", chunks[i].metadata["year"])


--- Chunk 1 ---
Dear shareholders, colleagues, customers, and partners, 

We are living through a time of historic challenge and opportunity. As I write this, the world faces ongoing economic, social, and geopolitical volatility. At the same time, we have entered a new age of AI that will fundamentally transform productivity for every individual, organization, and industry on earth, and help us address some of our most pressing challenges. 

This next generation of AI will reshape every software category and ev
Year: 2023

--- Chunk 2 ---
Amid this transformation, our mission to empower every person and every organization on the planet to achieve more remains constant. As a company, we believe we can be the democratizing force for this new generation of technology and the opportunity it will help unlock for every country, community, and individual, while mitigating its risks. 

Here are just a few examples of how we are already doing this: 

•	Leading electronic health records vendor 

#step3 embeddings

In [48]:
#!pip install -U sentence-transformers langchain-huggingface

In [49]:
from langchain_huggingface import HuggingFaceEmbeddings

In [50]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [51]:
text = chunks[0].page_content

vector = embeddings.embed_query(text)

print("Original text:")
print(text[:500])

print("\nVector length:", len(vector))
print("\nFirst 10 values:")
print(vector[:10])

Original text:
Dear shareholders, colleagues, customers, and partners, 

We are living through a time of historic challenge and opportunity. As I write this, the world faces ongoing economic, social, and geopolitical volatility. At the same time, we have entered a new age of AI that will fundamentally transform productivity for every individual, organization, and industry on earth, and help us address some of our most pressing challenges. 

This next generation of AI will reshape every software category and ev

Vector length: 384

First 10 values:
[-0.04844231903553009, -0.02738860249519348, 0.055798452347517014, -0.0007778463768772781, 0.06601578742265701, 0.0169635359197855, 0.014981215819716454, -0.03623655065894127, 0.01428846176713705, -0.01956944353878498]


In [52]:
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Total chunks:", len(chunks))
print("Total vectors:", len(vectors))
print("Vector length:", len(vectors[0]))

Total chunks: 1143
Total vectors: 1143
Vector length: 384


# vector db

In [55]:
!pip install -U faiss-cpu

In [53]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


In [54]:
print("Number of vectors:", vectorstore.index.ntotal)

Number of vectors: 1143


In [57]:
query = "What was the company's revenue in 2025?"

In [58]:
results = vectorstore.similarity_search(
    query,
    k=5
)

In [74]:
for i, doc in enumerate(results):

    print(f"\n===== Result {i+1} =====")

    print("Year:", doc.metadata.get("year"))
    print("Source:", doc.metadata.get("source"))

    print("\nContent:")
    print(doc.page_content[:1])


===== Result 1 =====
Year: 2025
Source: data/2025_AnnualReport.docx

Content:
2

===== Result 2 =====
Year: 2025
Source: data/2025_AnnualReport.docx

Content:
6

===== Result 3 =====
Year: 2023
Source: data/2023_Annual_Report.docx

Content:
Y

===== Result 4 =====
Year: 2025
Source: data/2025_AnnualReport.docx

Content:
(

===== Result 5 =====
Year: 2024
Source: data/2024_Annual_Report.docx

Content:
Y


#llm pipeline

In [69]:
#!pip install -U langchain-google-genai

In [70]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [75]:
API_KEY = open("api.txt", "r").read().strip()

print("API key loaded successfully!")

API key loaded successfully!


In [76]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=API_KEY
)

print("Gemini 3.8 Flash connected successfully!")

Gemini 3.8 Flash connected successfully!


In [77]:
question = "What was Microsoft's revenue in 2025?"

results = vectorstore.similarity_search(
    question,
    k=5
)

for i, doc in enumerate(results):
    print(f"\n--- RESULT {i+1} ---")
    print(doc.page_content[:1000])
    print("Metadata:", doc.metadata)


--- RESULT 1 ---
Operating Income

  

$

128,528

 

  

$

109,433

 

  

 

17%

 

 

  

 

 



  

 

 



  

 

 

 

Reportable Segments 

Fiscal Year 2025 Compared with Fiscal Year 2024 

Productivity and Business Processes 

Revenue increased $14.0 billion or 13%. 

•	Microsoft 365 Commercial products and cloud services revenue increased $10.8 billion or 14%. Microsoft 365 Commercial cloud revenue grew 15% with Microsoft 365 Commercial seat growth of 6% driven by small and medium businesses and frontline worker offerings, as well as growth in revenue per user. Microsoft 365 Commercial products revenue grew 7% driven by the Windows Commercial on-premises components of Microsoft 365 suite sales and an increase in Office transactional purchasing with the launch of Office 2024.
Metadata: {'source': 'data/2025_AnnualReport.docx', 'year': '2025'}

--- RESULT 2 ---
Highlights from fiscal year 2024 compared with fiscal year 2023 included: 

•	Microsoft Cloud revenue increased 23%

In [78]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [79]:
prompt = PromptTemplate(
    template="""
You are a financial report question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
- Do not use outside information.
- Do not guess.
- Give a direct answer.
- If the information is not present, say:
"I could not find this information in the provided reports."

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

parser = StrOutputParser()

chain = prompt | llm | parser

In [80]:
def ask_report(question, k=15):

    # 1. Retrieve relevant chunks from FAISS
    results = vectorstore.similarity_search(
        question,
        k=k
    )

    # 2. Combine retrieved chunks into context
    context = "\n\n".join(
        doc.page_content
        for doc in results
    )

    # 3. Send context and question to the chain
    answer = chain.invoke({
        "context": context,
        "question": question
    })

    # 4. Return clean answer
    return answer.strip()

In [81]:
print(
    ask_report(
        "What was Microsoft's total company revenue in fiscal year 2025?"
    )
)

$281,724 million (or $281.7 billion)


In [82]:
print(
    ask_report(
        "What was the revenue growth of More Personal Computing "
        "from fiscal year 2024 to fiscal year 2025?"
    )
)

Based on the provided report, the revenue for More Personal Computing was $62,032 million in fiscal year 2024 and $54,649 million in fiscal year 2025 (as shown in the segment tables for the 2025 vs. 2024 comparison). 

However, looking at the narrative for "Fiscal Year 2025 Compared with Fiscal Year 2024", the text states:
> "Revenue increased $36.6 billion or 15% with growth across each of our segments. Intelligent Cloud revenue increased driven by Azure. Productivity and Business Processes revenue increased driven by Microsoft 365 Commercial cloud. More Personal Computing revenue increased driven by Gaming and Search and news advertising."

It does not list a specific percentage change for More Personal Computing in fiscal year 2025 compared with fiscal year 2024. Therefore, I could not find the exact percentage growth rate for this segment in the provided reports.


In [83]:
print(ask_report(
    "What was Microsoft's total revenue in fiscal year 2024?"
))

$245,122 million


In [84]:
print(ask_report(
    "What was Microsoft's operating income in fiscal year 2025?"
))

$128,528 million


In [85]:
print(ask_report(
    "What was Microsoft's operating income in fiscal year 2025?"
))

$128,528 million


In [86]:
print(ask_report(
    "How much did Microsoft's total revenue increase from fiscal year 2024 to fiscal year 2025?"
))

Revenue increased $36.6 billion or 15% from fiscal year 2024 to fiscal year 2025.


In [87]:
print(ask_report(
    "What was Microsoft's total revenue in fiscal years 2023, 2024, and 2025?"
))

Based on the provided reports, Microsoft's total revenue was:

- **Fiscal Year 2025:** $281,724 million
- **Fiscal Year 2024:** $245,122 million
- **Fiscal Year 2023:** $211,915 million


In [88]:
print(ask_report(
    "What was Microsoft's total revenue in fiscal years 2023, 2024, and 2025?"
))

Based on the provided reports, Microsoft's total revenue was:

* **Fiscal Year 2025:** $281,724 million
* **Fiscal Year 2024:** $245,122 million
* **Fiscal Year 2023:** $211,915 million


In [89]:
print(ask_report(
    "What was Microsoft's revenue in fiscal year 2030?"
))

I could not find this information in the provided reports.


In [90]:
print(ask_report(
    "What was Microsoft's total revenue in fiscal years 2023, 2024, and 2025?"
))

Based on the provided reports, Microsoft's total revenue was:

* **Fiscal Year 2025:** $281,724 million ($281.7 billion)
* **Fiscal Year 2024:** $245,122 million ($245.1 billion)
* **Fiscal Year 2023:** $211,915 million ($211.9 billion)


In [91]:
print(ask_report(
    "What was Microsoft's total revenue in fiscal years 2023, 2024?"
))

Based on the provided reports, Microsoft's total revenue was:

* **Fiscal Year 2024:** $245,122 million ($245.1 billion)
* **Fiscal Year 2023:** $211,915 million ($211.9 billion)
